# Ablation 3 v6: Multi-Scale Training (Complete Zero-Augmentation)

**Research Question**: Open: does multi-scale training [416, 512, 640] improve wound segmentation across wound sizes.

**Intervention**: Train with random image sizes per batch, all other augmentations explicitly disabled.

**Fix from v5**: All 14+ augmentation parameters set explicitly so YOLOv8 defaults do not apply.
- v5 failure: only `auto_augment=None` was set, YOLOv8 used the `randaugment` default, Dice 4.8%
- v6 approach: all parameters set explicitly: hsv_h, hsv_s, hsv_v, translate, scale, fliplr, mosaic, erasing, auto_augment=None, and the rest

**Expected Outcome**: Dice +1-2% via improved handling of wound size distribution.

**Status**: Production deployment (3 seeds, 100 epochs, P100 GPU)


In [ ]:
# Lock NumPy 1.26.4 (ultralytics upgrades to 2.x, breaks OpenCV)
import subprocess
import sys

print("Installing ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

print("Force-reinstalling NumPy 1.26.4...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy==1.26.4"
], check=True)

import numpy as np
print(f"OK NumPy locked: {np.__version__}")
assert np.__version__.startswith("1.26"), f"NumPy version mismatch: {np.__version__}"


In [ ]:
import os
from pathlib import Path
import shutil

# Kaggle paths
IN_KAGGLE = os.path.exists("/kaggle/input")
BASE_INPUT_PATH = Path("/kaggle/input") if IN_KAGGLE else Path(".")
OUTPUT_PATH = Path("/kaggle/working") if IN_KAGGLE else Path(".")

# Dataset paths
DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs" / "data_wound_seg"
CODE_PATH = BASE_INPUT_PATH / "wound-seg-code-v1"

# Output directory
ABLATION_OUTPUT = OUTPUT_PATH / "ablation_3_v6_multiscale"
ABLATION_OUTPUT.mkdir(exist_ok=True, parents=True)

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset: {DATASET_PATH}")
print(f"Code: {CODE_PATH}")
print(f"Output: {ABLATION_OUTPUT}")


In [ ]:
# Copy dataset to working directory
WORKING_DATASET = OUTPUT_PATH / "data_wound_seg"

if not WORKING_DATASET.exists():
    print(f"Copying dataset from {DATASET_PATH}...")
    shutil.copytree(DATASET_PATH, WORKING_DATASET)
    print(f"OK Dataset copied to {WORKING_DATASET}")
else:
    print(f"OK Dataset already exists at {WORKING_DATASET}")

# Verify structure
required_dirs = ["train_images", "train_masks", "test_images", "test_masks"]
for d in required_dirs:
    path = WORKING_DATASET / d
    count = len(list(path.glob("*"))) if path.exists() else 0
    print(f"  {d}: {count} files")


In [ ]:
import sys
sys.path.insert(0, str(CODE_PATH))

print(f"OK Added {CODE_PATH} to Python path")
print(f"sys.path[0]: {sys.path[0]}")


In [ ]:
from ultralytics import YOLO
import yaml
import json
from datetime import datetime

print("OK Imports successful")


In [ ]:
# Create YOLO dataset configuration
config_path = OUTPUT_PATH / "wound_dataset.yaml"
config_content = f"""path: {WORKING_DATASET.parent}
train: data_wound_seg/train
val: data_wound_seg/test

names:
  0: wound
"""

with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"OK YOLO config created: {config_path}")

# Verify config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    print(f"  Dataset path: {config['path']}")
    print(f"  Train: {config['train']}")
    print(f"  Val: {config['val']}")


In [ ]:
# Training configuration
RANDOM_SEEDS = [42, 123, 456]
EPOCHS = 100
PATIENCE = 10

# P100-optimized hyperparameters (LOCKED from baseline)
BATCH = 32
LR0 = 0.015
WORKERS = 8
AMP = False

# Ablation 3: Multi-scale training
IMGSZ = [416, 512, 640]  # Random selection per batch

print("Training Configuration:")
print(f"  Seeds: {RANDOM_SEEDS}")
print(f"  Epochs: {EPOCHS} (patience: {PATIENCE})")
print(f"  Batch: {BATCH}")
print(f"  Learning rate: {LR0}")
print(f"  Workers: {WORKERS}")
print(f"  AMP: {AMP}")
print(f"  Image sizes (multi-scale): {IMGSZ}")


In [ ]:
# Multi-seed training with COMPLETE zero-augmentation
training_results = {}

for seed in RANDOM_SEEDS:
    print(f"\n{'='*60}")
    print(f"Training seed {seed}")
    print(f"{'='*60}")

    # Initialize model
    model = YOLO("yolov8n-seg.pt")

    # Train with multi-scale + COMPLETE zero-augmentation
    results = model.train(
        data=str(config_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,  # Multi-scale: [416, 512, 640]
        batch=BATCH,
        lr0=LR0,
        workers=WORKERS,
        patience=PATIENCE,
        amp=AMP,
        seed=seed,
        save=True,
        project=str(ABLATION_OUTPUT / "yolo_training"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False,

        # === COMPLETE ZERO-AUGMENTATION (14+ parameters) ===
        # Color/HSV augmentations
        hsv_h=0.0,          # Hue shift (default: 0.015)
        hsv_s=0.0,          # Saturation shift (default: 0.7)
        hsv_v=0.0,          # Value/brightness shift (default: 0.4)

        # Geometric augmentations
        degrees=0.0,        # Rotation (default: 0.0)
        translate=0.0,      # Translation (default: 0.1)
        scale=0.0,          # Scaling (default: 0.5)
        shear=0.0,          # Shear (default: 0.0)
        perspective=0.0,    # Perspective transform (default: 0.0)

        # Flip augmentations
        flipud=0.0,         # Vertical flip (default: 0.0)
        fliplr=0.0,         # Horizontal flip (default: 0.5)

        # Advanced augmentations
        mosaic=0.0,         # Mosaic (default: 1.0) - CRITICAL
        mixup=0.0,          # MixUp (default: 0.0)
        copy_paste=0.0,     # Copy-paste (default: 0.0)
        auto_augment=None,  # RandAugment (default: "randaugment") - CRITICAL
        erasing=0.0,        # Random erasing (default: 0.4)

        # Mosaic-related
        close_mosaic=0,     # Disable mosaic in final epochs (default: 10)
    )

    # Save best model path
    best_model_path = ABLATION_OUTPUT / "yolo_training" / f"seed_{seed}" / "weights" / "best.pt"
    training_results[seed] = {
        "best_model_path": str(best_model_path),
        "completed": datetime.now().isoformat()
    }

    print(f"OK Seed {seed} complete: {best_model_path}")

print(f"\n{'='*60}")
print(f"All {len(RANDOM_SEEDS)} seeds completed")
print(f"{'='*60}")


## args.yaml Verification

**Check**: YOLOv8 used the zero-augmentation parameters.

Ablation 3 v5 set `auto_augment=None`. YOLOv8 ignored it and used the `randaugment` default. The result was a 95% collapse.


In [ ]:
import yaml

print("\n=== AUGMENTATION PARAMETER VERIFICATION ===\n")

for seed in RANDOM_SEEDS:
    args_path = ABLATION_OUTPUT / "yolo_training" / f"seed_{seed}" / "args.yaml"

    with open(args_path, 'r') as f:
        args = yaml.safe_load(f)

    print(f"Seed {seed} - Checking augmentation parameters:")

    # Expected zero-augmentation parameters
    expected_params = {
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
        'degrees': 0.0, 'translate': 0.0, 'scale': 0.0,
        'shear': 0.0, 'perspective': 0.0,
        'flipud': 0.0, 'fliplr': 0.0,
        'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0,
        'erasing': 0.0, 'auto_augment': None
    }

    mismatches = []
    for param, expected in expected_params.items():
        actual = args.get(param)
        if actual != expected:
            mismatches.append(f"  FAIL {param}: expected {expected}, got {actual}")

    if mismatches:
        print("\nWARNING  WARNING: YOLOv8 IGNORED PARAMETERS:")
        for m in mismatches:
            print(m)
        print("\nALERT Training may have used unintended augmentations!")
    else:
        print("  OK All augmentation parameters correctly applied")

    # Verify multi-scale
    imgsz = args.get('imgsz')
    print(f"  Multi-scale imgsz: {imgsz}")
    if imgsz != [416, 512, 640]:
        print(f"    WARNING  Expected [416, 512, 640], got {imgsz}")

    print()


In [ ]:
# Save completion status
completion_status = {
    "notebook": "ablation_3_v6_multiscale",
    "status": "complete",
    "timestamp": datetime.now().isoformat(),
    "seeds": RANDOM_SEEDS,
    "configuration": {
        "intervention": "Multi-scale training [416, 512, 640]",
        "zero_augmentation": "ALL 14+ parameters explicitly set to 0.0 or None",
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "batch": BATCH,
        "lr0": LR0,
        "workers": WORKERS,
        "amp": AMP
    },
    "training_results": training_results
}

status_path = ABLATION_OUTPUT / "completion_status.json"
with open(status_path, 'w') as f:
    json.dump(completion_status, f, indent=2)

print(f"OK Completion status saved: {status_path}")
print(f"\nAblation 3 v6 complete!")
print(f"Next: Download outputs and run evaluation")
